# PREDICCIONES
Es una guía strip de 500nm de ancho por 220 nm de alto de silicio con cladding de óxido de silicio (abajo de la oblea) y aire.
- En esta simulación, el modo fundamental será TE.
- En cuanto a los modos, entiendo que 2 TE y 2 TM con estas dimensiones (no he hecho cálculos).
- El rango de neff estará entre 1.46 (aprox índice del óxido de silicio) y 3.42 (aprox índice del silicio) y crecerá si tenemos aire enxima en vez de óxido encima.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tidy3d as td
#Todo esto no consume créditos porque se ejecuta en local, lo que consume créditos son las llamadas a td.web

lda0 = 1.55 #micrómetros, Tidy3d trabja en micras
freq0 = td.C_0/lda0
w_core = 0.5
h_core = 0.22

MATERIALES

In [12]:
print(td.material_library["cSi"].variants.keys())
print(td.material_library["SiO2"].variants.keys())

dict_keys(['Palik_NoLoss', 'Palik_LowLoss', 'Palik_Lossy', 'SalzbergVilla1957', 'Li1993_293K', 'Green2008', 'Green2008_Lossless', 'Si_MultiPhysics'])
dict_keys(['Palik_NoLoss', 'Palik_LowLoss', 'Palik_Lossy', 'Horiba'])


In [41]:
si = td.material_library["cSi"]["Palik_NoLoss"]
siO2 = td.material_library["SiO2"]["Palik_NoLoss"]

In [42]:
print(si.nk_model(freq0)) #esto te devuelve el índice del material y la k en ese orden

(np.float64(3.477203636825235), np.float64(0.0))


In [43]:
print(siO2.nk_model(freq0))

(np.float64(1.4440347085541112), np.float64(0.0))


Escojo el Pakil_NoLoss que está en ambos y no quiero pérdidas porque a la longitud de onda a la que estamos trabajando ambos son transparentes. Como vemos k=0 en ambos, respaldando esto.

GEOMETRÍA.

In [45]:
#Esto es el box de silicio
guia = td.Structure(geometry=td.Box(center=(0,0,h_core/2),size=(td.inf,w_core,h_core)),medium=si)
guia

Structure(geometry=Box(center=(0.0, 0.0, 0.11), size=(inf, 0.5, 0.22)), name=None, background_permittivity=None, background_medium=None, priority=None,medium=cSi_PalikNoLoss)

In [46]:
#Esto es el box de SiO2
box = td.Structure(geometry=td.Box.from_bounds(rmin=(-td.inf,-td.inf,-4), rmax=(td.inf,td.inf,0)),medium=siO2)
box

Structure(geometry=Box(center=(0.0, 0.0, -2.0), size=(inf, inf, 4.0)), name=None, background_permittivity=None, background_medium=None, priority=None,medium=SiO2_Palik_NoLoss)

El otro medio lo asumo como aire entonces no hay que crear nada más.

SIMULACIÓN

In [63]:

print(f'Frecuencia de estudio {freq0*10**(-12)} THz')


Frecuencia de estudio 193.41448903225805 THz


In [ ]:
#Primero comprobemos si la frecuencia de los materiales cae dentro de la frecuencia de estudio:
f_min, f_max = si.frequency_range
print(f'{f_min*10**(-12)} THz')
print(f'{f_max*10**(-12)} THz')


49.965409666666666 THz
249.82704833333335 THz


In [67]:
f_min,f_max = siO2.frequency_range
print(f'{f_min*10**(-12)} THz')
print(f'{f_max*10**(-12)} THz')

59.9584916 THz
545.0771963636364 THz


Los dos rangos incluyen 193 THz

In [68]:
sim = td.Simulation(
    center=(0,0,h_core/2),
    size=(1,4,3), #Cuidado con este size, debe ser más grande que la guía, si no por principio de inertidumbre tienes más momento y más energía cinética (k mayor)
    grid_spec=td.GridSpec.auto(
        min_steps_per_wvl=20,
        wavelength=lda0),
    structures = [box,guia], #Aunque aquí no importe pon guia después porque se superpone lo que va después.
    run_time=1e-12
)

PLANO DE MODOS Y FIGURA DE SECCIÓN

In [69]:
from tidy3d.plugins.mode import ModeSolver